# AquaSynex Phase 2.3: Feature Engineering & Exploratory Analysis

**SIH26146 – AI-Powered Monitoring & Analysis of Bitcoin Transaction Traffic**

This notebook provides exploratory statistical analysis of the canonical feature matrix (`features_v1`)
generated from the clean canonical dataset (`database/aquasynex.duckdb`).

> **Notice**: In accordance with the ML audit protocol, **NO models are trained in this notebook**.
> The ground-truth label is loaded strictly as an exploratory grouping variable for descriptive statistical comparisons.

In [ ]:
import os
import duckdb
import numpy as np
import pandas as pd

# Connect to DuckDB analytical database
db_path = 'database/aquasynex.duckdb' if os.path.exists('database/aquasynex.duckdb') else '../database/aquasynex.duckdb'
con = duckdb.connect(db_path, read_only=True)
df_features = con.execute('SELECT * FROM features_v1').fetchdf()
df_labels = con.execute('SELECT txid as transaction_id, ground_truth_label, behavior_type FROM labels').fetchdf()
df_merged = df_features.merge(df_labels, on='transaction_id', how='left')
con.close()

print(f'[*] Loaded Feature Matrix: {df_features.shape[0]:,} rows x {df_features.shape[1]} columns')
print(f'[*] Merged with evaluation metadata: {df_merged.shape[0]:,} records')

In [ ]:
# Missingness and Infinite Value Audit
num_cols = df_features.select_dtypes(include=[np.number]).columns
nan_counts = df_features[num_cols].isna().sum()
inf_counts = np.isinf(df_features[num_cols]).sum()

print('=== DATA QUALITY AUDIT ===')
print(f'Total Features Analyzed: {len(num_cols)}')
print(f'Features with NaN values: {(nan_counts > 0).sum()}')
print(f'Features with Infinite values: {(inf_counts > 0).sum()}')
print(f'Duplicate Transaction IDs: {df_features["transaction_id"].duplicated().sum()}')

dtype_summary = df_features.dtypes.value_counts()
print('\nFeature Data Types:')
for dt, count in dtype_summary.items():
    print(f' - {dt}: {count} columns')

In [ ]:
# Compute comprehensive distribution statistics and skewness
stats_list = []
for col in num_cols:
    series = df_features[col]
    stats_list.append({
        'feature_name': col,
        'mean': series.mean(),
        'std': series.std(),
        'min': series.min(),
        'p25': series.quantile(0.25),
        'median': series.median(),
        'p75': series.quantile(0.75),
        'p95': series.quantile(0.95),
        'max': series.max(),
        'skewness': series.skew()
    })

df_stats = pd.DataFrame(stats_list)
print('=== FEATURE DISTRIBUTIONS & SKEWNESS (Top 10 High-Skew Features) ===')
print(df_stats.sort_values(by='skewness', ascending=False)[['feature_name', 'median', 'mean', 'p95', 'skewness']].head(10).to_string(index=False))

In [ ]:
# Feature Cardinality Analysis
cardinality = []
for col in df_features.columns:
    n_unique = df_features[col].nunique()
    cardinality.append({
        'feature_name': col,
        'unique_values': n_unique,
        'cardinality_type': 'Binary' if n_unique == 2 else ('Discrete' if n_unique < 50 else 'Continuous')
    })

df_card = pd.DataFrame(cardinality)
print('=== CARDINALITY BREAKDOWN ===')
print(df_card['cardinality_type'].value_counts())
print('\nDiscrete/Binary Features:')
print(df_card[df_card['cardinality_type'].isin(['Binary', 'Discrete'])][['feature_name', 'unique_values']].to_string(index=False))

In [ ]:
# Outlier Detection using Interquartile Range (IQR = Q3 - Q1)
outlier_summary = []
for col in num_cols:
    q1 = df_features[col].quantile(0.25)
    q3 = df_features[col].quantile(0.75)
    iqr = q3 - q1
    if iqr > 0:
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outliers = ((df_features[col] < lower) | (df_features[col] > upper)).sum()
        outlier_pct = (outliers / len(df_features)) * 100
        outlier_summary.append({
            'feature_name': col,
            'outlier_count': outliers,
            'outlier_pct': outlier_pct
        })

df_outliers = pd.DataFrame(outlier_summary)
print('=== TOP FEATURES WITH NOTABLE OUTLIERS ===')
print(df_outliers.sort_values(by='outlier_pct', ascending=False).head(10).to_string(index=False))

In [ ]:
# Correlation Matrix among Core Numerical Features
selected_cols = [
    'tx_input_count', 'tx_output_count', 'tx_total_output_sats', 'tx_fee_sats',
    'tx_fee_rate_sat_per_byte', 'tx_value_balance_ratio',
    'addr_hist_tx_count', 'addr_hist_active_days',
    'time_since_prev_global_tx_sec', 'time_since_prev_addr_tx_sec',
    'time_txs_last_1m', 'time_txs_last_5m',
    'rel_fan_in', 'rel_fan_out', 'rel_change_value_ratio'
]
corr_matrix = df_features[selected_cols].corr()
print('=== CORRELATION MATRIX SAMPLE ===')
print(np.round(corr_matrix[['tx_input_count', 'tx_output_count', 'tx_fee_sats', 'addr_hist_tx_count', 'time_txs_last_1m']], 2).to_string())

In [ ]:
# Descriptive Statistical Comparisons by Ground-Truth Group (Exploratory Only)
comparison_cols = [
    'tx_input_count', 'tx_output_count', 'tx_fee_rate_sat_per_byte',
    'addr_hist_tx_count', 'addr_reuse_count',
    'time_since_prev_addr_tx_sec', 'time_txs_last_1m',
    'rel_fan_in', 'rel_fan_out', 'rel_change_value_ratio'
]

grouped = df_merged.groupby('ground_truth_label')[comparison_cols].agg(['median', 'mean'])
print('=== DESCRIPTIVE COMPARISON: BENIGN (0) vs SUSPICIOUS (1) ===')
print(grouped.T.to_string())

In [ ]:
# Downstream Feature Categorization Matrix
print('=== DOWNSTREAM ELIGIBILITY SUMMARY ===')
print('Total Feature Matrix: 40 Features + 1 Transaction Identifier = 41 Columns')
print('Breakdown: 15 Transaction + 8 Address + 7 Temporal + 6 Network + 4 Relational')
print(' - Observational Categorical Context: net_country, net_asn (require categorical encoding before ML)')
print(' - Cyclical Preprocessing Note: time_hour_of_day, time_day_of_week should consider sin/cos transforms in modeling')
print(' - Retained Redundancy Note: tx_input_count/rel_fan_in, tx_output_count/rel_fan_out evaluated empirically in ML')
print(' - Candidates for Unsupervised Anomaly Detection (Isolation Forest): 18 dense continuous numerical features')
print(' - Candidates for Supervised Classification (XGBoost / LightGBM): All 40 features (with categorical encoding)')
print(' - Candidates for Graph Analysis (Phase 2.4): addr_hist_tx_count, rel_fan_in, rel_fan_out, addr_reuse_count')
print('\n[+] Phase 2.3 Feature Exploration successfully completed. Zero model training performed.')